#### TCP socket

In [ ]:
import socket

HOST = "0.0.0.0"
PORT = 5555


def handle_message(message: str) -> str:
    parts = message.strip().split("|")

    if len(parts) != 5 or parts[0] != "OBS":
        return "ERROR|invalid_message"

    vehicle_id = int(parts[1])
    x = float(parts[2])
    y = float(parts[3])
    speed = float(parts[4])

    print( f"Received: vehicle={vehicle_id}, " f"x={x}, y={y}, speed={speed}" )

    tx_power_mw = 17.5      # Temporary dummy action; no RL yet.

    return f"ACT|{vehicle_id}|{tx_power_mw}"


with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((HOST, PORT))
    server.listen(1)

    print(f"Python server listening on port {PORT}")

    conn, address = server.accept()

    with conn:
        print(f"Connected by {address}")

        reader = conn.makefile("r", encoding="utf-8")

        for line in reader:
            if not line.strip():
                continue

            try:
                response = handle_message(line)
            except (ValueError, IndexError) as exc:
                response = "ERROR|invalid_data"
                print(f"Invalid input: {exc}")

            conn.sendall((response + "\n").encode("utf-8"))
            print(f"Sent: {response}")


Python server listening on port 5555
Connected by ('172.24.162.72', 40232)
Received: vehicle=7, x=120.5, y=40.0, speed=13.2
Sent: ACT|7|17.5


In [ ]:
import json
import socketserver
import threading


HOST = "0.0.0.0"
PORT = 5555

# In the future, shared AgeNet state can be protected by this lock.
agent_lock = threading.Lock()


class AgeNetHandler(socketserver.StreamRequestHandler):
    def handle(self):
        print(f"[CONNECT] {self.client_address}")

        try:
            while True:
                line = self.rfile.readline()
                if not line: break

                try:
                    observation = json.loads(line.decode("utf-8"))
                    if observation.get("type") != "observe":
                        raise ValueError("Expected an observe message")

                    vehicle_id = observation["vehicle_id"]
                    with agent_lock:
                        self.process_observation(observation)

                        # Temporary dummy action; no RL yet.
                        response = { "type": "action", "vehicle_id": vehicle_id, "txPower_mW": 17.5, }

                    encoded = (json.dumps(response, separators=(",", ":")) + "\n").encode("utf-8")
                    self.wfile.write(encoded)
                    self.wfile.flush()

                except (ValueError, KeyError, TypeError) as exc:
                    print(f"[INVALID MESSAGE] {exc}")

                    response = {"type": "error", "message": "invalid_message",}
                    self.wfile.write((json.dumps(response) + "\n").encode("utf-8"))
                    self.wfile.flush()

        finally:
            print(f"[DISCONNECT] {self.client_address}")

    @staticmethod
    def process_observation(obs):
        vehicle_id = obs["vehicle_id"]
        sim_time = obs["sim_time"]
        position = obs["position"]
        neighbors = obs["neighbors"]

        print(
            f"[OBS] t={sim_time:.1f}s "
            f"vehicle={vehicle_id} "
            f"pos=({position['x']:.1f}, {position['y']:.1f}) "
            f"speed={obs['speed']:.2f} m/s "
            f"neighbors={len(neighbors)}"
        )

        for neighbor in neighbors:
            print(
                f"    -> {neighbor['vehicle_id']}: "
                f"distance={neighbor['distance']:.1f} m, "
                f"age={neighbor['age']:.2f} s"
            )


class ThreadedTCPServer( socketserver.ThreadingMixIn, socketserver.TCPServer, ):
    allow_reuse_address = True
    daemon_threads = True


if __name__ == "__main__":
    with ThreadedTCPServer((HOST, PORT), AgeNetHandler) as server:
        print(f"AgeNet bridge listening on {HOST}:{PORT}")
        server.serve_forever()


#### ZeroMQ 

In [ ]:
import zmq
import json

HOST = "0.0.0.0"
PORT = 5555

context = zmq.Context()
socket = context.socket(zmq.REP)
socket.bind(f"tcp://{HOST}:{PORT}")

print(f"Python ZeroMQ server listening on port {PORT}")
print("Waiting for C++ client...")

try:
    while True:
        message = socket.recv_json()
        print("Received from C++:", message)

        response = { "status": "ok", "action": 2, "received_state": message }   # Temporary dummy action; no RL yet.

        socket.send_json(response)
        print("Sent to C++:", response)

except KeyboardInterrupt:
    print("\nServer stopped.")

finally:
    socket.close()
    context.term()


Python ZeroMQ server listening on port 5555
Waiting for C++ client...
Received from C++: {'k': 3, 'r': 10, 'rho': 0.75, 'source': 'cpp_test'}
Sent to C++: {'status': 'ok', 'action': 2, 'received_state': {'k': 3, 'r': 10, 'rho': 0.75, 'source': 'cpp_test'}}
Received from C++: {'k': 3, 'r': 10, 'rho': 0.75, 'source': 'cpp_test'}
Sent to C++: {'status': 'ok', 'action': 2, 'received_state': {'k': 3, 'r': 10, 'rho': 0.75, 'source': 'cpp_test'}}
